# Production Data Pipeline — Notebook Walkthrough

Interactive companion to `pipeline.py`. The pipeline uses JSONL input, bronze/silver/gold layers, row validation, quarantine, deduplication by latest update, data-quality checks and source-hash idempotency.

In [ ]:
from pathlib import Path
import json, sqlite3, sys, tempfile

PROJECT = Path.cwd()
if PROJECT.name != 'production_data_pipeline':
    PROJECT = Path('projects/production_data_pipeline')
sys.path.insert(0, str(PROJECT.resolve()))
PROJECT.resolve()

In [ ]:
from pipeline import quality_checks, run_pipeline, seed_fixture

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    source = tmp / 'orders.jsonl'
    database = tmp / 'orders.sqlite'
    seed_fixture(source)
    first = run_pipeline(source, database)
    second = run_pipeline(source, database)
    with sqlite3.connect(database) as conn:
        checks = quality_checks(conn)
        gold = conn.execute('SELECT * FROM gold_daily_order_metrics ORDER BY order_date').fetchall()

print('first run:', json.dumps(first.as_dict(), indent=2))
print('second run skipped:', second.skipped)
print('quality checks:', checks)
print('gold rows:', gold)

In [ ]:
assert first.raw_rows == 5
assert first.quarantined_rows == 1
assert second.skipped is True
assert all(value == 0 for value in checks.values())
print('Pipeline checks passed.')

## What this proves

The same source file can be rerun safely without duplicating records, invalid rows are quarantined instead of silently loaded, the latest valid order update wins, and gold reporting tables are rebuilt only from clean silver data.